In [2]:
import pandas as pd
import joblib
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

from xgboost import XGBClassifier

# =========================================================
# LOAD DATASET
# =========================================================

DATASET_PATH = "Expanded_Cybersecurity_Dataset_50000.csv"

df = pd.read_csv(DATASET_PATH)

print("=" * 70)
print("DATASET LOADED")
print("=" * 70)
print("Shape:", df.shape)

# =========================================================
# REMOVE DUPLICATES
# =========================================================

before = len(df)

df.drop_duplicates(inplace=True)

after = len(df)

print(f"\nDuplicates Removed: {before-after}")

# =========================================================
# HANDLE MISSING VALUES
# =========================================================

categorical_cols = [
    "Industry",
    "Cloud_Usage",
    "Firewall_Installed",
    "EDR_Installed",
    "SIEM_Installed",
    "Compliance_Requirement",
    "Main_Threat_Concern",
    "Security_Budget",
    "Risk_Level"
]

for col in categorical_cols:

    if col in df.columns:

        df[col] = (
            df[col]
            .fillna("Unknown")
            .astype(str)
        )

df["Employees"] = pd.to_numeric(
    df["Employees"],
    errors="coerce"
)

df["Employees"] = (
    df["Employees"]
    .fillna(df["Employees"].median())
)

# =========================================================
# REMOVE EMPTY TARGETS
# =========================================================

df = df.dropna(
    subset=["Recommended_Solution"]
)

# =========================================================
# FEATURES
# =========================================================

feature_cols = [
    "Industry",
    "Employees",
    "Cloud_Usage",
    "Firewall_Installed",
    "EDR_Installed",
    "SIEM_Installed",
    "Compliance_Requirement",
    "Main_Threat_Concern",
    "Security_Budget",
    "Risk_Level"
]

X = df[feature_cols]

y = df["Recommended_Solution"].astype(str)

print("\nUnique Recommendation Classes:", y.nunique())

# =========================================================
# LABEL ENCODING
# =========================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

# =========================================================
# PREPROCESSING
# =========================================================

categorical_features = [
    "Industry",
    "Cloud_Usage",
    "Firewall_Installed",
    "EDR_Installed",
    "SIEM_Installed",
    "Compliance_Requirement",
    "Main_Threat_Concern",
    "Security_Budget",
    "Risk_Level"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

# =========================================================
# IMPROVED XGBOOST
# =========================================================

xgb_model = XGBClassifier(
    objective="multi:softprob",

    num_class=len(
        label_encoder.classes_
    ),

    n_estimators=1200,

    max_depth=8,

    learning_rate=0.05,

    subsample=0.9,

    colsample_bytree=0.9,

    min_child_weight=1,

    gamma=0.1,

    reg_alpha=0.1,

    reg_lambda=1.0,

    tree_method="hist",

    random_state=42,

    eval_metric="mlogloss",

    n_jobs=-1
)

# =========================================================
# PIPELINE
# =========================================================

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

# =========================================================
# TRAINING
# =========================================================

print("\n" + "=" * 70)
print("TRAINING STARTED")
print("=" * 70)

pipeline.fit(
    X_train,
    y_train
)

# =========================================================
# TEST PREDICTIONS
# =========================================================

preds = pipeline.predict(
    X_test
)

# =========================================================
# METRICS
# =========================================================

accuracy = accuracy_score(
    y_test,
    preds
)

precision = precision_score(
    y_test,
    preds,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_test,
    preds,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_test,
    preds,
    average="weighted",
    zero_division=0
)

print("\n" + "=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nClassification Report\n")

print(
    classification_report(
        y_test,
        preds,
        zero_division=0
    )
)

# =========================================================
# CROSS VALIDATION
# =========================================================

print("\nRunning Cross Validation...\n")

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    pipeline,
    X,
    y_encoded,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

print("=" * 70)
print("CROSS VALIDATION")
print("=" * 70)

print("Scores:", cv_scores)

print(
    "Mean Accuracy:",
    round(cv_scores.mean(), 4)
)

# =========================================================
# FEATURE IMPORTANCE
# =========================================================

print("\nExtracting Feature Importance...")

pipeline.fit(X, y_encoded)

model = pipeline.named_steps["model"]

feature_names = (
    pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

importance = model.feature_importances_

feature_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

feature_df = feature_df.sort_values(
    by="Importance",
    ascending=False
)

print("\nTOP 20 IMPORTANT FEATURES\n")

print(
    feature_df.head(20)
)

feature_df.to_csv(
    "feature_importance.csv",
    index=False
)

# =========================================================
# SAVE MODEL
# =========================================================

joblib.dump(
    pipeline,
    "recommendation_model.pkl"
)

joblib.dump(
    label_encoder,
    "label_encoder.pkl"
)

print("\n" + "=" * 70)
print("FILES SAVED")
print("=" * 70)

print("recommendation_model.pkl")
print("label_encoder.pkl")
print("feature_importance.csv")

print("\nTRAINING COMPLETED SUCCESSFULLY")

DATASET LOADED
Shape: (50000, 13)

Duplicates Removed: 0

Unique Recommendation Classes: 20

Train Shape: (40000, 10)
Test Shape : (10000, 10)

TRAINING STARTED

MODEL PERFORMANCE
Accuracy  : 0.0501
Precision : 0.0500
Recall    : 0.0501
F1 Score  : 0.0500

Classification Report

              precision    recall  f1-score   support

           0       0.06      0.06      0.06       502
           1       0.05      0.05      0.05       496
           2       0.05      0.05      0.05       497
           3       0.05      0.04      0.05       496
           4       0.04      0.04      0.04       493
           5       0.05      0.04      0.05       499
           6       0.05      0.06      0.05       503
           7       0.03      0.03      0.03       494
           8       0.06      0.06      0.06       505
           9       0.07      0.07      0.07       498
          10       0.05      0.05      0.05       504
          11       0.05      0.05      0.05       491
          12     

In [ ]:
print(df["Recommended_Solution"].nunique())